In [1]:
import pandas as pd

df = pd.read_excel(
    "../data/processed/CarbonOpt_IsolationForest.xlsx"
)

print("Shape:", df.shape)
print(df.columns.tolist())
print(df.head())
print(df.dtypes)
print(df.isnull().sum())

Shape: (3162, 10)
['state', 'year', 'residential_billion_btu', 'commercial_billion_btu', 'industrial_billion_btu', 'transportation_billion_btu', 'co2_million_metric_tons', 'co2_per_energy', 'energy_growth_rate', 'co2_growth_rate']
  state  year  residential_billion_btu  commercial_billion_btu  \
0    AK  1962                    10472                    7129   
1    AK  1963                    11256                    9246   
2    AK  1964                    13004                   12021   
3    AK  1965                    13857                   14210   
4    AK  1966                    15563                   15066   

   industrial_billion_btu  transportation_billion_btu  \
0                   24859                       34182   
1                   25598                       32387   
2                   25522                       32246   
3                   22874                       34379   
4                   33619                       36233   

   co2_million_metric_tons  c

In [2]:
X = df[features].copy()

print(X.shape)
print(X.head())

NameError: name 'features' is not defined

In [3]:
features = [
    "residential_billion_btu",
    "commercial_billion_btu",
    "industrial_billion_btu",
    "transportation_billion_btu",
    "co2_million_metric_tons",
    "co2_per_energy",
    "energy_growth_rate",
    "co2_growth_rate"
]
X = df[features].copy()

print(X.shape)
print(X.head())

(3162, 8)
   residential_billion_btu  commercial_billion_btu  industrial_billion_btu  \
0                    10472                    7129                   24859   
1                    11256                    9246                   25598   
2                    13004                   12021                   25522   
3                    13857                   14210                   22874   
4                    15563                   15066                   33619   

   transportation_billion_btu  co2_million_metric_tons  co2_per_energy  \
0                       34182                    5.327        0.000070   
1                       32387                    5.366        0.000068   
2                       32246                    5.620        0.000068   
3                       34379                    5.720        0.000067   
4                       36233                    6.812        0.000068   

   energy_growth_rate  co2_growth_rate  
0            9.457298         9.925

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(3162, 8)


In [5]:
from sklearn.ensemble import IsolationForest

isolation_model = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

isolation_model.fit(X_scaled)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",300
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.05
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [6]:
df["anomaly_prediction"] = isolation_model.predict(X_scaled)

In [7]:
df["anomaly_status"] = df["anomaly_prediction"].map({
    1: "Normal",
    -1: "Anomaly"
})

In [8]:
print(df["anomaly_status"].value_counts())

anomaly_status
Normal     3003
Anomaly     159
Name: count, dtype: int64


In [9]:
df["anomaly_score"] = isolation_model.decision_function(X_scaled)

In [10]:
anomalies = df[
    df["anomaly_status"] == "Anomaly"
].sort_values("anomaly_score")

print(
    anomalies[
        [
            "state",
            "year",
            "co2_million_metric_tons",
            "co2_per_energy",
            "energy_growth_rate",
            "co2_growth_rate",
            "anomaly_score",
            "anomaly_status"
        ]
    ].head(20)
)

     state  year  co2_million_metric_tons  co2_per_energy  energy_growth_rate  \
2724    TX  2020                  626.644        0.000048           -5.402447   
2722    TX  2018                  681.981        0.000050            6.278864   
2726    TX  2022                  667.641        0.000048            0.830911   
2725    TX  2021                  664.004        0.000048            5.696167   
2727    TX  2023                  672.456        0.000048            2.269129   
2723    TX  2019                  682.830        0.000050            1.351856   
2717    TX  2013                  657.257        0.000053            6.340598   
2719    TX  2015                  652.844        0.000052            0.918056   
2721    TX  2017                  662.035        0.000052            0.968283   
2720    TX  2016                  653.535        0.000052            1.187677   
2718    TX  2014                  661.270        0.000054           -0.267498   
2713    TX  2009            

In [11]:
from pathlib import Path

MODEL_DIR = Path("../models")
OUTPUT_DIR = Path("../outputs/isolation_forest")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import joblib

joblib.dump(
    isolation_model,
    MODEL_DIR / "isolation_forest.pkl"
)

joblib.dump(
    scaler,
    MODEL_DIR / "isolation_scaler.pkl"
)


['..\\models\\isolation_scaler.pkl']

In [12]:
df.to_csv(
    OUTPUT_DIR / "anomaly_results.csv",
    index=False
)

In [1]:
print("Total observations:", len(df))
print("Anomalies detected:", (df["anomaly"] == -1).sum())
print("Normal observations:", (df["anomaly"] == 1).sum())
print("Anomaly percentage:", (df["anomaly"] == -1).mean() * 100)

NameError: name 'df' is not defined

In [2]:
import pandas as pd

df = pd.read_excel("../data/processed/CarbonOpt_IsolationForest.xlsx")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (3162, 10)

Columns:
['state', 'year', 'residential_billion_btu', 'commercial_billion_btu', 'industrial_billion_btu', 'transportation_billion_btu', 'co2_million_metric_tons', 'co2_per_energy', 'energy_growth_rate', 'co2_growth_rate']


In [3]:
import pandas as pd
import joblib

# Load dataset
df = pd.read_excel("../data/processed/CarbonOpt_IsolationForest.xlsx")

# Features used by Isolation Forest
isolation_features = [
    "residential_billion_btu",
    "commercial_billion_btu",
    "industrial_billion_btu",
    "transportation_billion_btu",
    "co2_million_metric_tons",
    "co2_per_energy",
    "energy_growth_rate",
    "co2_growth_rate"
]

# Load saved scaler and model
scaler = joblib.load("../models/isolation_scaler.pkl")
isolation_model = joblib.load("../models/isolation_forest.pkl")

# Transform features
X = df[isolation_features]
X_scaled = scaler.transform(X)

# Generate anomaly labels
df["anomaly"] = isolation_model.predict(X_scaled)

# Calculate statistics
total = len(df)
anomalies = (df["anomaly"] == -1).sum()
normal = (df["anomaly"] == 1).sum()
anomaly_percentage = anomalies / total * 100

print("ISOLATION FOREST RESULTS")
print("------------------------")
print(f"Total observations : {total}")
print(f"Normal observations: {normal}")
print(f"Anomalies detected  : {anomalies}")
print(f"Anomaly percentage  : {anomaly_percentage:.2f}%")

ISOLATION FOREST RESULTS
------------------------
Total observations : 3162
Normal observations: 3003
Anomalies detected  : 159
Anomaly percentage  : 5.03%


In [4]:
# Stage 1D — Inspect Isolation Forest anomalies

import pandas as pd
import joblib

# Load dataset
df = pd.read_excel("../data/processed/CarbonOpt_IsolationForest.xlsx")

# Features used by Isolation Forest
isolation_features = [
    "residential_billion_btu",
    "commercial_billion_btu",
    "industrial_billion_btu",
    "transportation_billion_btu",
    "co2_million_metric_tons",
    "co2_per_energy",
    "energy_growth_rate",
    "co2_growth_rate"
]

# Load model and scaler
scaler = joblib.load("../models/isolation_scaler.pkl")
isolation_model = joblib.load("../models/isolation_forest.pkl")

# Transform features
X = df[isolation_features]
X_scaled = scaler.transform(X)

# Predict anomaly labels
df["anomaly"] = isolation_model.predict(X_scaled)

# Separate anomalies
anomalies = df[df["anomaly"] == -1].copy()

print("ANOMALY INSPECTION")
print("------------------")
print(f"Number of anomalies: {len(anomalies)}")

print("\nAverage feature values:")
print(anomalies[isolation_features].mean().round(3))

print("\nAverage feature values — normal observations:")
print(df[df["anomaly"] == 1][isolation_features].mean().round(3))

ANOMALY INSPECTION
------------------
Number of anomalies: 159

Average feature values:
residential_billion_btu       1094679.874
commercial_billion_btu         962413.849
industrial_billion_btu        3004353.195
transportation_billion_btu    2027235.270
co2_million_metric_tons           379.839
co2_per_energy                      0.000
energy_growth_rate                  0.133
co2_growth_rate                    -0.250
dtype: float64

Average feature values — normal observations:
residential_billion_btu       286460.586
commercial_billion_btu        211494.084
industrial_billion_btu        463198.747
transportation_billion_btu    354110.517
co2_million_metric_tons           81.540
co2_per_energy                     0.000
energy_growth_rate                 1.483
co2_growth_rate                    1.241
dtype: float64
